# World News Video Pipeline (Colab + GPU)

Ye notebook aapko Google Colab par poora project **run** karne deta hai:

- News fetch → script → voice (Edge TTS) → visuals (OpenAI images ya local) → MP4 render
- Optional: **SadTalker** lip-sync / talking anchor (GPU recommended)
- Optional: YouTube upload (OAuth)

## Drive folder structure

Google Drive me ye structure banayein:

- `NewsBot/project/`  → is repo ka code
- `NewsBot/secrets/`  → `client_secret.json`
- `NewsBot/assets/anchor/` → anchor images (1.png…)
- `NewsBot/assets/news_today/` → optional daily visuals (01_*.jpg…)
- `NewsBot/assets/news_music/` → background music tracks

**Important:** `.env` me keys paste karni hoti hain. Keys ko public mat karein.

## 0) Runtime

Colab me:
- **Runtime → Change runtime type → GPU** (lip-sync chahiye to)
- Agar sirf normal video test karna hai (static anchor + AI images), GPU optional hai.

In [ ]:
# 1) Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/NewsBot'
PROJECT_DIR = f'{DRIVE_ROOT}/project'
SECRETS_DIR = f'{DRIVE_ROOT}/secrets'
ASSETS_DIR = f'{DRIVE_ROOT}/assets'

print('PROJECT_DIR:', PROJECT_DIR)
print('SECRETS_DIR:', SECRETS_DIR)
print('ASSETS_DIR:', ASSETS_DIR)

In [ ]:
# 2) Put repo code into Drive (choose ONE)
# Option A: Upload zip to Drive and extract into $PROJECT_DIR (manual)
# Option B: If your repo is on GitHub, clone it:

# %cd /content
# !rm -rf "$PROJECT_DIR"
# !git clone https://github.com/<you>/<repo>.git "$PROJECT_DIR"

# For this local Cursor workspace, easiest is: copy your project folder into Drive/NewsBot/project manually.

!ls -la "$PROJECT_DIR" || echo "Project folder not found. Create Drive/NewsBot/project and copy code."

In [ ]:
# 3) Install system deps
!sudo apt-get update -y
!sudo apt-get install -y ffmpeg

In [ ]:
# 4) Python deps
%cd "$PROJECT_DIR"
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt

In [ ]:
# 5) Secrets & assets linking
# We will use Drive folders for secrets and assets.

%cd "$PROJECT_DIR"

# Ensure local folders exist
!mkdir -p secrets assets/anchor assets/news_today assets/news_music

# Copy OAuth client secret (downloaded from Google Cloud) into repo secrets/
!cp -f "$SECRETS_DIR/client_secret.json" secrets/client_secret.json 2>/dev/null || echo "Put client_secret.json into Drive/NewsBot/secrets/"

# Sync assets from Drive into repo
!rsync -a --delete "$ASSETS_DIR/anchor/" assets/anchor/ 2>/dev/null || true
!rsync -a --delete "$ASSETS_DIR/news_today/" assets/news_today/ 2>/dev/null || true
!rsync -a --delete "$ASSETS_DIR/news_music/" assets/news_music/ 2>/dev/null || true

!ls -la assets/anchor | head -20
!ls -la assets/news_today | head -20
!ls -la assets/news_music | head -20

In [ ]:
# 6) Configure .env (python cell)
# Edit values below, then run this cell.

# If you want 'free tier' like your friend's setup, set SCRIPT_PROVIDER/SEO_PROVIDER to gemini.
GEMINI_API_KEY = ""  # paste here (Google AI Studio)
OPENAI_API_KEY = ""  # optional (leave blank if using gemini)

SCRIPT_PROVIDER = "gemini"  # gemini | openai
SEO_PROVIDER = "gemini"     # gemini | openai

PEXELS_API_KEY = ""   # optional
BROLL_SOURCE = "local"  # openai | pexels | local

EDGE_TTS_VOICE = "hi-IN-SwaraNeural"  # Hindi voice
TARGET_DURATION_SEC = 900

SEO_METADATA_USE_LLM = 1
LIPSYNC_ENABLE = 0

BACKGROUND_MUSIC_VOLUME = 0.12

env_text = f"""GEMINI_API_KEY={GEMINI_API_KEY}
OPENAI_API_KEY={OPENAI_API_KEY}

SCRIPT_PROVIDER={SCRIPT_PROVIDER}
SEO_PROVIDER={SEO_PROVIDER}

BROLL_SOURCE={BROLL_SOURCE}
PEXELS_API_KEY={PEXELS_API_KEY}
PEXELS_WHEN_EMPTY=1

SEO_METADATA_USE_LLM={SEO_METADATA_USE_LLM}
OPENAI_SEO_MODEL=gpt-4o-mini
GEMINI_SCRIPT_MODEL=gemini-1.5-flash
GEMINI_SEO_MODEL=gemini-1.5-flash

OPENAI_IMAGE_MODEL=dall-e-3
OPENAI_IMAGE_SIZE=1792x1024

LIPSYNC_ENABLE={LIPSYNC_ENABLE}
SADTALKER_DIR=
SADTALKER_PYTHON=

TARGET_DURATION_SEC={TARGET_DURATION_SEC}
EDGE_TTS_VOICE={EDGE_TTS_VOICE}

BACKGROUND_MUSIC_VOLUME={BACKGROUND_MUSIC_VOLUME}

ANCHOR_IMAGE_DIR=./assets/anchor
NEWS_MEDIA_DIR=./assets/news_today
NEWS_MUSIC_DIR=./assets/news_music
OUTPUT_DIR=./output
TEMP_DIR=./temp

YOUTUBE_CLIENT_SECRETS=./secrets/client_secret.json
YOUTUBE_TOKEN_PICKLE=./secrets/youtube_token.pickle
"""

with open('.env', 'w', encoding='utf-8') as f:
    f.write(env_text)

print('Wrote .env (keys hidden)')
print('\n'.join(
    (line.split('=')[0] + '=***REDACTED***')
    if line.startswith(('OPENAI_API_KEY=', 'PEXELS_API_KEY=', 'GEMINI_API_KEY='))
    else line
    for line in env_text.splitlines()[:60]
))

In [ ]:
# 7) Preflight + generate video (no upload)
%cd "$PROJECT_DIR"
!python -m src.main --preflight
!python -m src.main --skip-upload --privacy unlisted

## 8) (Optional) Upload to YouTube

- Ensure `secrets/client_secret.json` is present.
- Run:

```bash
python -m src.main --privacy unlisted
```

First run opens an auth link / browser prompt in Colab. Approve access.
Token will be stored at `secrets/youtube_token.pickle` inside Drive project folder.

Tip: Start with `unlisted`, then switch to `public` once satisfied.

## 9) (Optional) SadTalker lip-sync (GPU)

This is **optional** and heavy. Steps summary:

- Clone SadTalker into Drive (example path: `NewsBot/SadTalker/`)
- Install its dependencies + download weights (one-time)
- Set in `.env`:
  - `LIPSYNC_ENABLE=1`
  - `SADTALKER_DIR=/content/drive/MyDrive/NewsBot/SadTalker`

Then run `python -m src.main --skip-upload` again.

Note: free Colab sessions expire; for reliable daily production you’ll want a GPU VM.

In [ ]:
# 9a) (Optional) SadTalker quick setup (advanced)
# WARNING: This downloads large weights. Run only if you understand Colab GPU limits.

# %cd "$DRIVE_ROOT"
# !rm -rf SadTalker
# !git clone https://github.com/OpenTalker/SadTalker.git SadTalker
# %cd "$DRIVE_ROOT/SadTalker"
# !pip install -q -r requirements.txt
#
# # Download weights per SadTalker instructions (varies by repo version)
# # After setup, update .env in $PROJECT_DIR:
# #   LIPSYNC_ENABLE=1
# #   SADTALKER_DIR=/content/drive/MyDrive/NewsBot/SadTalker
#
# %cd "$PROJECT_DIR"
# !python -m src.main --skip-upload